# Hard-row marks reader (specialist)
**GPU (T4/P100) on, Internet on.** Upload the new `kaggle_marks.zip` as a new dataset version first.

What this trains: the current ResNet-CTC, warm-started from `marks_sum.pt`, on the real rows (train students only) with the **hard rows oversampled**, plus realistic synthetic hard rows (a|b pairs, sums, grader ticks across the column line, scribbled-out entries) built only from train-sheet digits. Early stopping on the validation students; the test students are scored once at the end. Each run prints a train-vs-validation gap so you can see overfitting.

Baseline to beat on the held-out rows (current model): 90% all, 99% simple, ~46-54% hard.

### 1. Setup (prints `True True True`)

In [ ]:
import glob, os, shutil, subprocess
os.makedirs('/kaggle/working/proj', exist_ok=True)
# Kaggle usually auto-extracts an uploaded zip, so look for the extracted folder first, then for the zip itself
hits = glob.glob('/kaggle/input/**/scripts/train_marks_sum.py', recursive=True)
if hits:
    src = os.path.dirname(os.path.dirname(hits[0]))
    shutil.copytree(src, '/kaggle/working/proj', dirs_exist_ok=True)
    print('copied extracted dataset from', src)
else:
    z = glob.glob('/kaggle/input/**/kaggle_marks.zip', recursive=True) + glob.glob('/kaggle/working/kaggle_marks.zip')
    assert z, 'Dataset not found: use "Add Input" on the right and attach your marks dataset, then re-run this cell.'
    subprocess.run(['unzip', '-qo', z[0], '-d', '/kaggle/working/proj'], check=True)
    print('unzipped', z[0])
os.chdir('/kaggle/working/proj')
print(os.path.exists('scripts/train_marks_hard.py'), os.path.exists('data/cells.npz'), os.path.exists('models/marks.pt'))

### 2. GPU check

In [ ]:
!nvidia-smi | head -15

### 3. Run A: specialist, no position channels (seed 0)

In [ ]:
!cd /kaggle/working/proj && python -u scripts/train_marks_hard.py --epochs 80 --seed 0 --out models/hard_a_s0.pt 2>&1 | grep --line-buffered -v -i warn | tee hard_a_s0.log

### 4. Run B: same + a|b position channels (`--coord`, seed 0)

In [ ]:
!cd /kaggle/working/proj && python -u scripts/train_marks_hard.py --epochs 80 --seed 0 --coord --out models/hard_b_s0.pt 2>&1 | grep --line-buffered -v -i warn | tee hard_b_s0.log

### 5. Compare A and B (paste this output to Claude)

In [ ]:
!echo '=== A (no coord) ===' && grep -B1 -A12 'best checkpoint' hard_a_s0.log
!echo '=== B (coord) ===' && grep -B1 -A12 'best checkpoint' hard_b_s0.log

### 6. Two more seeds of the better one (change `--coord` to match the winner; remove it if A won)
Stability check: if seeds disagree a lot the gain is noise.

In [ ]:
!cd /kaggle/working/proj && for s in 1 2; do python -u scripts/train_marks_hard.py --epochs 80 --seed $s --coord --out models/hard_final_s$s.pt 2>&1 | grep --line-buffered -v -i warn | tee hard_final_s$s.log; done

In [ ]:
!grep -h -A8 'VALIDATION' hard_final_s1.log hard_final_s2.log

### 7. Download the models
Run, then use the Output panel (`/kaggle/working`) to download `marks_hard_models.zip`.

In [ ]:
import shutil, os
os.chdir('/kaggle/working')
shutil.make_archive('/kaggle/working/marks_hard_models', 'zip', '/kaggle/working/proj/models')
print(round(os.path.getsize('marks_hard_models.zip')/1e6), 'MB')
for f in ['hard_a_s0.log','hard_b_s0.log','hard_final_s1.log','hard_final_s2.log']:
    if os.path.exists('/kaggle/working/proj/'+f): shutil.copy('/kaggle/working/proj/'+f, '/kaggle/working/'+f)